# ✨ LivePortrait Studio - Lái Chuyển Động Cho Ảnh Nhân Vật (Tự Động Lưu Vào Google Drive)
> **Hướng dẫn 1-Click (Không cần tải lại lần sau):**
> 1. Bấm nút **'Sao chép vào Drive'** ở thanh trên cùng để lưu vĩnh viễn notebook này vào Google Drive của bạn.
> 2. Bấm **Thời gian chạy (Runtime)** -> **Thay đổi loại phần cứng** -> Chọn **T4 GPU**.
> 3. Bấm **Chạy tất cả (Run all)** -> Bấm **'Kết nối với Google Drive'** khi được hỏi.
> 4. Lần đầu sẽ tải ~2GB trọng số và lưu vào Drive, **từ lần thứ 2 trở đi sẽ nạp tức thì từ Drive mà KHÔNG CẦN TẢI LẠI!**

In [ ]:
#@title Chạy LivePortrait WebUI 1-Click (Tự Động Caching Google Drive)
import os
import shutil
from google.colab import drive
from IPython.display import clear_output

print("🔗 Đang kết nối với Google Drive của bạn...")
drive.mount('/content/drive')

drive_cache_dir = "/content/drive/MyDrive/AI_Colab_Cache/LivePortrait"
drive_weights_dir = f"{drive_cache_dir}/pretrained_weights"
os.makedirs("/content/drive/MyDrive/AI_Colab_Cache", exist_ok=True)

%cd /content
if not os.path.exists("/content/LivePortrait"):
    !git clone -b dev https://github.com/camenduru/LivePortrait /content/LivePortrait

%cd /content/LivePortrait

# Kiểm tra xem 2GB weights đã có trên Drive chưa
if os.path.exists(drive_weights_dir) and os.path.isdir(drive_weights_dir) and len(os.listdir(drive_weights_dir)) > 3:
    print("🎉 ĐÃ TÌM THẤY 2GB TRỌNG SỐ TRONG GOOGLE DRIVE! Nạp trực tiếp không cần tải lại...")
    !rm -rf /content/LivePortrait/pretrained_weights
    !cp -r "{drive_weights_dir}" /content/LivePortrait/pretrained_weights
else:
    print("⏳ Đang tải trọng số AI lần đầu (~2GB) và lưu vào Google Drive của bạn...")
    !rm -rf /content/LivePortrait/pretrained_weights
    !git clone https://huggingface.co/camenduru/LivePortrait /content/LivePortrait/pretrained_weights
    print("💾 Đang lưu bản sao trọng số vào Google Drive để lần sau nạp ngay lập tức...")
    os.makedirs(drive_cache_dir, exist_ok=True)
    !cp -r /content/LivePortrait/pretrained_weights "{drive_cache_dir}/"

# Cài đặt thư viện (Không cố định phiên bản cũ để tương thích với Python mới của Colab)
!pip install -q tyro onnxruntime-gpu onnx gradio colorama ffmpeg-python

%cd /content/LivePortrait/src/utils/dependencies/insightface/thirdparty/face3d/mesh/cython
!python setup.py build_ext --inplace

%cd /content/LivePortrait
clear_output()
print("🚀 Đang khởi động LivePortrait WebUI...")
!python app.py --share
